# SAIA Python — Demo Notebook

This notebook demonstrates all public functions of the `saia_python` wrapper for the GWDG SAIA platform.

**API key — three options (pick one):**

1. **Environment variable** — set `SAIA_API_KEY` before launching Jupyter
2. **`.saia_api` file** — create a file in this directory (or your home dir) containing just the key:
   ```
   e939a2158b8a63048a6aa241294f523e
   ```
3. **`.env` file** — create a file in this directory (or your home dir) with:
   ```
   SAIA_API_KEY=e939a2158b8a63048a6aa241294f523e
   ```

`SAIAClient()` (no arguments) tries all three in the order listed above.

In [ ]:
import os
import sys

# Add parent dir so saia_python is importable without installation
sys.path.insert(0, os.path.abspath(".."))

from saia_python import load_api_key, SAIAClient

# SAIAClient with all parameters shown (defaults resolve automatically)
client = SAIAClient(
    api_key=None,            # explicit key, or None for auto-discovery, default None
    base_url=None,           # explicit URL, or None for config.toml / hardcoded default, default None
    key_file=None,           # explicit path to .saia_api or .env file, default None
)
client

## 1. OOP Interface — SAIAClient

### 1.1 List Models

In [ ]:
model_ids = client.models.list_ids()
print(f"Available models ({len(model_ids)}):")
for mid in model_ids:
    print(f"  - {mid}")

In [ ]:
# Full model details
models = client.models.list()
models[:2]

In [ ]:
# Probe which models support tool calling (one API call per model — may take a few minutes)
# tool_models = client.models.list_tool_capable(
#     verbose=False,       # True to print per-model results
# )
# print(f"\nTool-capable models ({len(tool_models)}):")
# for m in tool_models:
#     print(f"  - {m}")

### 1.2 Chat Completion

In [ ]:
response = client.chat.completions(
    model="meta-llama-3.1-8b-instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of Germany?"},
    ],
    temperature=0.3,         # sampling temperature (0–2), default None
    top_p=None,              # nucleus sampling (0–1), default None
    max_tokens=50,           # maximum tokens to generate, default None
    stream=False,            # True for streaming generator, default False
)

print("Assistant:", response["choices"][0]["message"]["content"])

### 1.3 Streaming Chat Completion

In [ ]:
stream = client.chat.completions(
    model="meta-llama-3.1-8b-instruct",
    messages=[{"role": "user", "content": "Count from 1 to 5."}],
    temperature=None,        # default None
    top_p=None,              # default None
    max_tokens=50,           # default None
    stream=True,             # returns a generator yielding chunks
)

print("Streaming: ", end="")
for chunk in stream:
    choices = chunk.get("choices", [])
    if not choices:
        continue
    content = choices[0].get("delta", {}).get("content", "")
    print(content, end="", flush=True)
print()

### 1.4 Rate Limits

In [ ]:
rate_limits = client.get_rate_limits()
print(rate_limits)

In [ ]:
# Rate limits are also available on chat responses from previous chat completions call
# This saves you an extra API call if you want to check your limits after a response
print(response["_rate_limits"])

### 1.5 Voice AI

Requires an audio file (WAV, MP3, MP4, or FLAC). Uncomment and adjust the path to test. 
Current limit of 600s.

In [ ]:
# Transcription
# transcript = client.voice.transcribe(
#     "path/to/audio.wav",
#     model="whisper-large-v2",    # default "whisper-large-v2"
#     response_format="text",      # "text", "vtt", or "srt", default "text"
#     language="de",               # language hint (e.g. "de", "en"), default None
#     wait=True,                   # False for fire-and-forget, default True
# )
# print("Transcript:", transcript)

In [ ]:
# Translation to English
# translation = client.voice.translate(
#     "path/to/audio.wav",
#     model="whisper-large-v2",    # default "whisper-large-v2"
#     response_format="text",      # "text", "vtt", or "srt", default "text"
#     wait=True,                   # False for fire-and-forget, default True
# )
# print("Translation:", translation)

### 1.6 Arcanas

In [ ]:
from saia_python import load_arcana_ids

# Load ARCANA IDs and show full summary
arcana_ids = load_arcana_ids()
print(client.arcana.summary(arcana_ids=arcana_ids))

# Extract the default ID for use in subsequent cells
default_id = arcana_ids.get("default", "")

In [ ]:
from saia_python import load_arcana_ids

# Load ARCANA IDs from env var / .env file
arcana_ids = load_arcana_ids()
print(f"Found {len(arcana_ids)} ARCANA ID(s):")
for label, aid in arcana_ids.items():
    print(f"  {label}: {aid}")

# The ID is "owner/name" — extract just the name for management endpoints
default_id = arcana_ids.get("default", "")
arcana_name = default_id.split("/", 1)[1] if "/" in default_id else default_id
print(f"\nDefault ARCANA ID: {default_id}")
print(f"Arcana name (for get/upload): {arcana_name}")
print(f"Arcana ID   (for RAG chat):   {default_id}")

# List all arcanas available on the server
arcanas = client.arcana.list()
print(f"\nAvailable arcanas on server ({len(arcanas)}):")
for a in arcanas:
    print(f"  - {a['name']} (owner: {a['owner_user_name']}, files: {a['file_count']})")

In [ ]:
default_id

In [ ]:
# Arcana details — accepts either "owner/name" or just "name"
if default_id:
    print(client.arcana.info(
        default_id,
        data=None,           # pre-fetched dict from get(), default None
        verbose=False,       # True for CLI/DB version info, default False
    ))
    print("\n--- verbose ---\n")
    print(client.arcana.info(default_id, verbose=True))
else:
    print("No ARCANA_ID configured — skipping")

In [ ]:
# List files in the arcana
if default_id:
    files = client.arcana.list_files(default_id)
    print(f"Files in arcana ({len(files)}):")
    for f in files[:5]:
        print(f"  - {f['name']} ({f['size']} bytes)")
    if len(files) > 5:
        print(f"  ... and {len(files) - 5} more")

#### Create and Delete Arcanas

In [ ]:
# Create a new arcana
# result = client.arcana.create(
#     "MyNewArcana",
#     append_uuid=True,        # True: append UUID4 suffix (default), False: use exact name
#     update_toml=False,       # True: add to config.toml after creation, default False
#     toml_label=None,         # label in [saia.arcana.labels], or None for ids array, default None
# )
# print(f"Created: {result['name']}")
# print(f"Full ID: {result['id']}")

# Create and register in config.toml under a label
# result = client.arcana.create(
#     "ProjectArcana", update_toml=True, toml_label="my_project"
# )

In [ ]:
# Delete an arcana entirely
# client.arcana.delete(
#     "saiauser123/MyNewArcana-xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx",
#     update_toml=False,       # True: remove from config.toml after deletion, default False
# )

#### ARCANA File and Index Management

In [ ]:
# Upload a single file
# client.arcana.upload(
#     default_id,
#     "path/to/document.pdf",
#     overwrite=False,         # True to replace existing file, default False
# )

In [ ]:
# Upload all files from a directory
# client.arcana.upload_directory(
#     default_id,
#     "path/to/docs/",
#     pattern="*",            # glob pattern, default "*" (all files)
#     recursive=False,        # True to include subdirectories, default False
#     overwrite=False,        # True to replace existing files, default False
#     verbose=False,          # True for per-file status output, default False
# )

# Example: upload only PDFs recursively with overwrite
# client.arcana.upload_directory(
#     default_id, "path/to/docs/", pattern="*.pdf", recursive=True, overwrite=True
# )

In [ ]:
# Delete a single file by name (use list_files to find the name)
# client.arcana.delete_file(default_id, "document.pdf")

# Delete all files matching a local directory (by filename)
# client.arcana.delete_directory(
#     default_id,
#     "path/to/docs/",
#     pattern="*",            # glob pattern, default "*"
#     recursive=False,        # True to include subdirectories, default False
#     verbose=False,          # True for per-file status output, default False
# )

In [ ]:
# Trigger re-indexing
# result = client.arcana.generate_index(
#     default_id,
#     wait=True,               # True: poll until done, False: fire-and-forget, default True
#     timeout=600,             # max seconds to wait (wait=True only), default 600
#     poll_interval=5,         # seconds between status checks, default 5
# )
# print(client.arcana.info(data=result))

# Fire-and-forget alternative
# client.arcana.generate_index(default_id, wait=False)
# print(client.arcana.info(default_id))  # check status manually

In [ ]:
# Delete the entire index
# client.arcana.delete_index(default_id)

In [ ]:
# Chat with RAG context (uses the full owner/name ID)
# if default_id:
#     rag_response = client.arcana.chat(
#         model="llama-3.3-70b-instruct",
#         messages=[{"role": "user", "content": "Summarize the uploaded document."}],
#         arcana_id=default_id,
#         temperature=None,        # sampling temperature (0–2), default None
#         max_tokens=None,         # maximum tokens to generate, default None
#         stream=False,            # True for streaming generator, default False
#     )
#     print(rag_response["choices"][0]["message"]["content"])

### 1.7 Document Conversion (Docling)

Convert PDF and other documents to markdown, HTML, JSON, or tokens.

In [ ]:
# Convert a PDF to markdown (full parameters)
# result = client.documents.convert(
#     "path/to/document.pdf",
#     response_type="markdown",           # "markdown", "html", "json", or "tokens", default "markdown"
#     extract_tables_as_images=None,       # True to render tables as images, default None
#     image_resolution_scale=None,         # 1–4 image quality multiplier, default None
# )
# print(result)                           # preview: filename, format, char count
# print(result.content[:500])             # first 500 chars of content
# result.save("output.md")               # save to file

In [ ]:
# Convenience: convert directly to HTML string
# html = client.documents.convert_to_html("path/to/document.pdf")
# print(html[:300])

# Convenience: convert directly to markdown string
# md = client.documents.convert_to_markdown("path/to/document.pdf")
# print(md[:300])

In [ ]:
# Advanced: tables as images + high resolution
# result = client.documents.convert(
#     "path/to/document.pdf",
#     response_type="markdown",
#     extract_tables_as_images=True,
#     image_resolution_scale=4,
# )
# print(f"{len(result.images)} images extracted")
# result.save("output_hires.md")

### 1.8 OpenAI SDK Compatibility

Use the OpenAI Python SDK directly against SAIA. Requires `pip install saia-python[openai]`.
This enables integration with tools like RAGAS, LangChain, and instructor.

In [ ]:
from saia_python import create_openai_client

# Standalone factory — credentials resolved automatically
openai_client = create_openai_client(
    api_key=None,            # explicit key, or None for auto-discovery, default None
    base_url=None,           # explicit URL, or None for config.toml / hardcoded default, default None
    key_file=None,           # explicit path to .saia_api or .env, default None
    async_client=False,      # True for AsyncOpenAI, default False
)
openai_client

In [ ]:
# Chat completions via OpenAI SDK
response = openai_client.chat.completions.create(
    model="llama-3.3-70b-instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Answer concisely."},
        {"role": "user", "content": "What is the GWDG?"},
    ],
    max_tokens=100,
)
print(response.choices[0].message.content)

In [ ]:
# Embeddings via OpenAI SDK
embedding = openai_client.embeddings.create(
    model="e5-mistral-7b-instruct",     # also: multilingual-e5-large-instruct, qwen3-embedding-4b
    input="The GWDG provides AI services for research.",
    encoding_format="float",             # default "float"
)
print(f"Embedding model: {embedding.model}")
print(f"Dimensions: {len(embedding.data[0].embedding)}")
print(f"First 5 values: {embedding.data[0].embedding[:5]}")

In [ ]:
# Alternative: access via SAIAClient property (same credentials, lazy-initialized)
# client.openai.chat.completions.create(model="...", messages=[...])
# client.openai_async  # for async workflows

---
## 2. Functional Interface

All services are also available as standalone functions.

In [ ]:
from saia_python import (
    list_model_ids,
    list_models,
    chat_completion,
    get_rate_limits,
    # transcribe,
    # translate,
    list_arcanas,
    # get_arcana,
    # upload_to_arcana,
    # arcana_chat,
)

# All functional API functions use auto-discovery for the API key
# (same resolution as SAIAClient: env var -> .saia_api -> .env).
# Pass api_key=... explicitly to override.

In [ ]:
# List model IDs
ids = list_model_ids()
print(f"{len(ids)} models available")
ids[:5]

In [ ]:
# Chat completion (functional API — full parameters)
resp = chat_completion(
    model="llama-3.3-70b-instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Answer concisely."},
        {"role": "user", "content": "Say hello in German, French, and Japanese."},
    ],
    temperature=None,            # default None
    top_p=None,                  # default None
    max_tokens=100,              # default None
    stream=False,                # default False
    api_key=None,                # default None (auto-discover)
    base_url=None,               # default None (from config.toml or hardcoded)
)
print(resp["choices"][0]["message"]["content"])

In [ ]:
# Rate limits
limits = get_rate_limits()
print(limits)

In [ ]:
# List arcanas
arcana_list = list_arcanas()
print(f"{len(arcana_list)} arcanas available")

---
## 3. Error Handling

In [ ]:
from saia_python import AuthenticationError, RateLimitError, APIError

# Example: catch authentication errors
try:
    bad_client = SAIAClient(api_key="invalid-key")
    bad_client.models.list_ids()
except AuthenticationError as e:
    print(f"Auth error (expected): {e}")
except APIError as e:
    print(f"API error: {e} (status {e.status_code})")

---
## 4. Tool Use (Function Calling)

The SAIA API is OpenAI-compatible and supports tool calling. This lets the model
decide when to call a function you define, and you execute it locally.

The pattern is:
1. Define tools as JSON schemas and send them with the chat request.
2. The model responds with a `tool_calls` list instead of a text message.
3. You execute the function locally and send the result back.
4. The model incorporates the result into its final answer.

In [ ]:
import json

# Step 1: Define a tool (a simple weather lookup)
def get_weather(city: str) -> dict:
    """Simulate a weather API call."""
    fake_data = {
        "Berlin": {"temp_c": 18, "condition": "partly cloudy"},
        "London": {"temp_c": 14, "condition": "rainy"},
        "Tokyo":  {"temp_c": 26, "condition": "sunny"},
    }
    return fake_data.get(city, {"temp_c": 20, "condition": "unknown"})

# OpenAI-compatible tool schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name, e.g. 'Berlin'.",
                    }
                },
                "required": ["city"],
            },
        },
    }
]

# Map of function names to callables
tool_dispatch = {"get_weather": get_weather}

print("Tool defined:", tools[0]["function"]["name"])

In [ ]:
# Step 2: Send a message that should trigger the tool
# Use a model that supports tool calling (e.g. llama-3.3-70b-instruct)
messages = [
    {"role": "user", "content": "What is the weather like in Berlin?"},
]

response = client.chat.completions(
    model="llama-3.3-70b-instruct",
    messages=messages,
    tools=tools,
    max_tokens=200,
)

assistant_message = response["choices"][0]["message"]
print("Model response:")
print(json.dumps(assistant_message, indent=2))

In [ ]:
# Step 3: Execute the tool call(s) and send results back to the model
tool_calls = assistant_message.get("tool_calls", [])

if not tool_calls:
    # Model answered directly without calling a tool
    print("No tool call — model answered directly:")
    print(assistant_message.get("content", ""))
else:
    # Append the assistant's tool-call message to the conversation
    messages.append(assistant_message)

    # Execute each tool call and append the result
    for tc in tool_calls:
        fn_name = tc["function"]["name"]
        fn_args = json.loads(tc["function"]["arguments"])
        print(f"Calling {fn_name}({fn_args})")

        result = tool_dispatch[fn_name](**fn_args)
        print(f"  -> {result}")

        messages.append({
            "role": "tool",
            "tool_call_id": tc["id"],
            "content": json.dumps(result),
        })

    # Step 4: Send the conversation (with tool results) back to the model
    final_response = client.chat.completions(
        model="llama-3.3-70b-instruct",
        messages=messages,
        tools=tools,
        max_tokens=200,
    )

    print("\nFinal answer:")
    print(final_response["choices"][0]["message"]["content"])